# Transform and Load
- Récupère les données collectées depuis S3
- Rassemble les données enrichies dans un fichier unique
- Charge les données dans la base de données

In [ ]:
# initialisation
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path
import boto3
import json
import os
import re
load_dotenv()

# definition de constantes
USER_AGENT = os.getenv("USER_AGENT")
AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_KEY")
AWS_REGION = os.getenv("AWS_REGION")
AWS_BUCKET = os.getenv("AWS_BUCKET")
AWS_BUCKET_DIR = os.getenv("AWS_BUCKET_DIR")
LOCAL_DIR = "./outputs/"

## Chargement données hotels pour regroupement

In [ ]:
s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name="eu-west-3",
)

In [ ]:
# chargement données depuis s3
# fichiers json hotels_infos.json et hotels_locations.json

obj_infos = s3_client.get_object(
    Bucket=AWS_BUCKET,
    Key=AWS_BUCKET_DIR + "hotels/hotels_infos.json"
)
hotels_infos = json.loads(obj_infos['Body'].read().decode('utf-8'))
print(hotels_infos[:3])

obj_locations = s3_client.get_object(
    Bucket=AWS_BUCKET,
    Key=AWS_BUCKET_DIR + "hotels/hotels_locations.json"
)
hotels_locations = json.loads(obj_locations['Body'].read().decode('utf-8'))
print(hotels_locations[:3])

In [ ]:
len(hotels_infos)

### fusion hotels_infos et hotels_locations

In [ ]:
def extract_slug(url):
    """
    Extrait le slug de l'URL de hotel
    """
    match = re.search(r'/hotel/[a-z]{2}/([^/]+)\.fr\.html', url)
    return match.group(1) if match else None

In [ ]:
df_infos = pd.DataFrame(hotels_infos)
df_infos['slug'] = df_infos['url'].apply(extract_slug)
 
df_locations = pd.DataFrame(hotels_locations)
 
df_merged = df_infos.merge(
    df_locations[['city_id', 'slug', 'latitude', 'longitude']],
    on=['city_id', 'slug'],
    how='left'
)

df_merged.head()

In [ ]:
len(df_merged)

In [ ]:
df_hotels_enriched = df_merged.drop(columns=['slug'])

df_hotels_enriched['score'] = df_hotels_enriched['score'].str.replace(',', '.').astype(float)
df_hotels_enriched['city_id'] = df_hotels_enriched['city_id'].astype('Int16')
df_hotels_enriched['votes'] = df_hotels_enriched['votes'].astype(float)

In [ ]:
# TODO: supprimer cette cellule (sauvegarde CSV+S3 supprimée, insertion directe en DB)

In [ ]:
df_hotels_enriched.dtypes

## Sauvegarde en BDD

In [ ]:
from sqlalchemy import create_engine, text

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
engine = create_engine(f"postgresql+psycopg://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")

try:
    with engine.begin() as conn:
        # supprime les données existantes avant insertion
        conn.execute(text("TRUNCATE TABLE hotels_enriched RESTART IDENTITY"))
        df_hotels_enriched.to_sql('hotels_enriched', conn, if_exists='append', index=False)
    print(f"{len(df_hotels_enriched)} hôtels insérés en base")
except Exception as e:
    print(f"Erreur insertion hotels_enriched: {e}")

In [ ]:
pd.__version__

In [ ]:
df_hotels_enriched.head()